# NNCE Actionability Benchmark Table

Notebook ini membentuk tabel ringkasan dari `evaluation/reports/experiments/nnce_actionability_benchmark.json`.
Kolom nama skenario sengaja tidak ditampilkan; setiap baris diberi ID `ACT-01` sampai `ACT-10`.

In [35]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import HTML, display

REPORT_PATH = Path("../reports/experiments/nnce_actionability_benchmark.json")

with REPORT_PATH.open(encoding="utf-8") as file:
    report = json.load(file)

len(report["scenarios"])

10

## Ringkasan Metrik Utama

In [36]:
summary = report["summary"]

summary_table = pd.DataFrame(
    [
        {
            "Metode": "NNCE Murni",
            "Tingkat Pelanggaran Fitur Immutable": summary["pure_nnce"]["immutable_violation_rate"],
            "Tingkat Pelanggaran Mutable di Luar Pilihan": summary["pure_nnce"]["outside_selected_mutable_violation_rate"],
            "Rata-rata Jumlah Fitur Immutable Berubah": summary["pure_nnce"]["average_changed_immutable_feature_count_all_scenarios"],
        },
        {
            "Metode": "NNCE Actionability",
            "Tingkat Pelanggaran Fitur Immutable": summary["adapted_nnce"]["immutable_violation_rate"],
            "Tingkat Pelanggaran Mutable di Luar Pilihan": summary["adapted_nnce"]["outside_selected_mutable_violation_rate"],
            "Rata-rata Jumlah Fitur Immutable Berubah": 0.0,
        },
    ]
)

display(HTML(summary_table.to_html(index=False)))

Metode,Tingkat Pelanggaran Fitur Immutable,Tingkat Pelanggaran Mutable di Luar Pilihan,Rata-rata Jumlah Fitur Immutable Berubah
NNCE Murni,1.0,0.4,1.0
NNCE Actionability,0.0,0.0,0.0


## Tabel Per Skenario dengan ID ACT

In [37]:
def yes_no(value: bool) -> str:
    return "Ya" if value else "Tidak"


def feature_list(features: list[str]) -> str:
    return ", ".join(features) if features else "-"


rows = []
for index, scenario in enumerate(report["scenarios"], start=1):
    pure = scenario["pure_nnce"]
    adapted = scenario["adapted_nnce"]
    rows.append(
        {
            "ID": f"ACT-{index:02d}",
            "Fitur yang Boleh Diubah": feature_list(scenario["mutable_allowed"]),
            "NNCE Murni: Pelanggaran Immutable": yes_no(pure["immutable_violation"]),
            "NNCE Murni: Pelanggaran Mutable di Luar Pilihan": yes_no(
                pure["outside_selected_mutable_violation"]
            ),
            "Fitur Immutable yang Berubah": feature_list(pure["changed_immutable_features"]),
            "Fitur Mutable di Luar Pilihan yang Berubah": feature_list(
                pure["changed_outside_selected_mutable_features"]
            ),
            "NNCE Actionability: Pelanggaran Immutable": yes_no(adapted["immutable_violation"]),
            "NNCE Actionability: Pelanggaran Mutable di Luar Pilihan": yes_no(
                adapted["outside_selected_mutable_violation"]
            ),
        }
    )

actionability_table = pd.DataFrame(rows)
display(HTML(actionability_table.to_html(index=False)))

ID,Fitur yang Boleh Diubah,NNCE Murni: Pelanggaran Immutable,NNCE Murni: Pelanggaran Mutable di Luar Pilihan,Fitur Immutable yang Berubah,Fitur Mutable di Luar Pilihan yang Berubah,NNCE Actionability: Pelanggaran Immutable,NNCE Actionability: Pelanggaran Mutable di Luar Pilihan
ACT-01,BMI,Ya,Tidak,age,-,Tidak,Tidak
ACT-02,"BMI, is_hypertension",Ya,Ya,age,is_cholesterol,Tidak,Tidak
ACT-03,"BMI, is_cholesterol",Ya,Ya,age,smoking_status,Tidak,Tidak
ACT-04,"BMI, moderate_physical_activity_frequency, is_hypertension",Ya,Tidak,age,-,Tidak,Tidak
ACT-05,"BMI, moderate_physical_activity_frequency, is_hypertension, is_cholesterol",Ya,Tidak,age,-,Tidak,Tidak
ACT-06,BMI,Ya,Ya,age,is_hypertension,Tidak,Tidak
ACT-07,"BMI, smoking_status",Ya,Tidak,age,-,Tidak,Tidak
ACT-08,"BMI, is_hypertension, smoking_status",Ya,Ya,age,"is_cholesterol, moderate_physical_activity_frequency",Tidak,Tidak
ACT-09,"BMI, moderate_physical_activity_frequency, is_hypertension, smoking_status",Ya,Tidak,age,-,Tidak,Tidak
ACT-10,"smoking_status, BMI, moderate_physical_activity_frequency, is_hypertension, is_cholesterol",Ya,Tidak,age,-,Tidak,Tidak


## Projection Ablation: HEOM Semua Fitur vs HEOM Fitur Boleh Diubah

Tabel ini membandingkan hasil full projection ketika neighbor dipilih dengan HEOM seluruh fitur dan HEOM yang hanya menghitung fitur yang boleh diubah.

In [38]:
FULL_SPACE_HEOM_REPORT_PATH = Path(
    "../reports/ablation/nn_projection/projection_ablation_full_space_heom_report.json"
)
MUTABLE_HEOM_REPORT_PATH = Path(
    "../reports/ablation/nn_projection/projection_ablation_report.json"
)

with FULL_SPACE_HEOM_REPORT_PATH.open(encoding="utf-8") as file:
    full_space_heom_report = json.load(file)

with MUTABLE_HEOM_REPORT_PATH.open(encoding="utf-8") as file:
    mutable_heom_report = json.load(file)

len(full_space_heom_report["profiles"]), len(mutable_heom_report["profiles"])

(30, 30)

### Tabel Per Profil

In [39]:
full_space_profiles = {
    profile["profile_id"]: profile for profile in full_space_heom_report["profiles"]
}
mutable_profiles = {
    profile["profile_id"]: profile for profile in mutable_heom_report["profiles"]
}

shared_profile_ids = [
    profile["profile_id"]
    for profile in mutable_heom_report["profiles"]
    if profile["profile_id"] in full_space_profiles
]

projection_rows = []
for index, profile_id in enumerate(shared_profile_ids, start=1):
    full_space = full_space_profiles[profile_id]["full_projection"]
    mutable = mutable_profiles[profile_id]["full_projection"]
    projection_rows.append(
        {
            "ID": f"HEOM-{index:02d}",
            "Proximity HEOM Semua Fitur": full_space["proximity_normalized_l1"],
            "Proximity HEOM Fitur Boleh Diubah": mutable["proximity_normalized_l1"],
            "Sparsity HEOM Semua Fitur": full_space["changed_feature_count"],
            "Sparsity HEOM Fitur Boleh Diubah": mutable["changed_feature_count"],
        }
    )

projection_table = pd.DataFrame(projection_rows)
display(HTML(projection_table.to_html(index=False)))

ID,Proximity HEOM Semua Fitur,Proximity HEOM Fitur Boleh Diubah,Sparsity HEOM Semua Fitur,Sparsity HEOM Fitur Boleh Diubah
HEOM-01,0.533617,0.520412,1,1
HEOM-02,0.324750,0.317342,1,2
HEOM-03,0.394249,0.304615,2,2
HEOM-04,0.323097,0.305716,1,1
HEOM-05,0.256871,0.239911,1,1
HEOM-06,0.197504,0.105774,2,1
HEOM-07,1.089915,1.089915,2,2
HEOM-08,1.049856,1.049856,2,2
HEOM-09,0.427441,0.279361,1,1
HEOM-10,0.125717,0.067981,1,1


### Tabel Agregat

In [40]:
projection_summary_table = pd.DataFrame(
    [
        {
            "Metrik": "Mean Proximity",
            "HEOM Semua Fitur": full_space_heom_report["summary"]["full_mean_proximity"],
            "HEOM Fitur Boleh Diubah": mutable_heom_report["summary"]["full_mean_proximity"],
        },
        {
            "Metrik": "Mean Sparsity",
            "HEOM Semua Fitur": full_space_heom_report["summary"]["full_mean_changed_feature_count"],
            "HEOM Fitur Boleh Diubah": mutable_heom_report["summary"]["full_mean_changed_feature_count"],
        },
    ]
)

display(HTML(projection_summary_table.to_html(index=False)))

Metrik,HEOM Semua Fitur,HEOM Fitur Boleh Diubah
Mean Proximity,0.431184,0.310487
Mean Sparsity,1.533333,1.200000


## Projection Ablation: Full Projection vs Sparse Projection

Tabel ini membandingkan full projection dengan sparse projection pada kandidat valid target probability.

In [41]:
SPARSE_PROJECTION_REPORT_PATH = Path(
    "../reports/ablation/nn_projection/projection_ablation_best_candidate_report.json"
)

with SPARSE_PROJECTION_REPORT_PATH.open(encoding="utf-8") as file:
    sparse_projection_report = json.load(file)

len(sparse_projection_report["profiles"])

30

### Tabel Per Profil

In [42]:
sparse_rows = []
for index, profile in enumerate(sparse_projection_report["profiles"], start=1):
    full = profile["full_projection"]
    sparse = profile["prefix_sparse_projection"]
    sparse_rows.append(
        {
            "ID": f"SP-{index:02d}",
            "Proximity Full Projection": full["proximity_normalized_l1"],
            "Proximity Sparse Projection": sparse["proximity_normalized_l1"],
            "Sparsity Full Projection": full["changed_feature_count"],
            "Sparsity Sparse Projection": sparse["changed_feature_count"],
        }
    )

sparse_projection_table = pd.DataFrame(sparse_rows)
display(HTML(sparse_projection_table.to_html(index=False)))

ID,Proximity Full Projection,Proximity Sparse Projection,Sparsity Full Projection,Sparsity Sparse Projection
SP-01,0.520412,0.520412,1,1
SP-02,0.317342,0.317342,2,2
SP-03,0.304615,0.304615,2,2
SP-04,0.305716,0.305716,1,1
SP-05,0.239911,0.239911,1,1
SP-06,0.105774,0.105774,1,1
SP-07,1.089915,1.089915,2,2
SP-08,1.049856,1.049856,2,2
SP-09,0.279361,0.279361,1,1
SP-10,0.067981,0.067981,1,1


### Tabel Agregat

In [43]:
sparse_projection_summary_table = pd.DataFrame(
    [
        {
            "Metrik": "Mean Proximity",
            "Full Projection": sparse_projection_report["summary"]["full_mean_proximity"],
            "Sparse Projection": sparse_projection_report["summary"]["sparse_mean_proximity"],
        },
        {
            "Metrik": "Mean Sparsity",
            "Full Projection": sparse_projection_report["summary"]["full_mean_changed_feature_count"],
            "Sparse Projection": sparse_projection_report["summary"]["sparse_mean_changed_feature_count"],
        },
    ]
)

display(HTML(sparse_projection_summary_table.to_html(index=False)))

Metrik,Full Projection,Sparse Projection
Mean Proximity,0.310487,0.277601
Mean Sparsity,1.200000,1.166667


## Objective Weight Sweep

Tabel ini merangkum proximity dan plausibility (LOF) kandidat terpilih pada setiap konfigurasi bobot objective.

In [44]:
OBJECTIVE_WEIGHT_SWEEP_REPORT_PATH = Path(
    "../reports/ablation/nn_projection/objective_weight_sweep.json"
)

with OBJECTIVE_WEIGHT_SWEEP_REPORT_PATH.open(encoding="utf-8") as file:
    objective_weight_sweep_report = json.load(file)

len(objective_weight_sweep_report["profiles"])

30

### Tabel Proximity Per Profil

In [45]:
objective_config_columns = [
    ("proximity_only", "Proximity Only"),
    ("proximity_75_plausibility_25", "75/25"),
    ("proximity_50_plausibility_50", "50/50"),
    ("proximity_25_plausibility_75", "25/75"),
    ("plausibility_only", "Plausibility Only"),
]

proximity_rows = []
for index, profile in enumerate(objective_weight_sweep_report["profiles"], start=1):
    selections = profile["selected_candidates"]
    row = {"ID": f"PROX-{index:02d}"}
    for config_key, column_name in objective_config_columns:
        row[column_name] = selections[config_key]["proximity"]
    proximity_rows.append(row)

objective_proximity_table = pd.DataFrame(proximity_rows)
display(HTML(objective_proximity_table.to_html(index=False)))

ID,Proximity Only,75/25,50/50,25/75,Plausibility Only
PROX-01,0.520412,0.520412,0.520412,0.588438,0.588438
PROX-02,0.324750,0.350197,0.350197,0.350197,0.350197
PROX-03,0.385338,0.385338,0.385338,0.385338,0.385338
PROX-04,0.305716,0.305716,0.306822,0.306822,0.306822
PROX-05,0.239911,0.239911,0.385420,0.385420,0.385420
PROX-06,0.105774,0.106568,0.106568,0.106568,0.106568
PROX-07,1.089915,1.093228,1.093228,1.093228,1.093228
PROX-08,1.049856,1.059038,1.059038,1.059038,1.059038
PROX-09,0.279361,0.279361,0.279361,0.339980,0.919138
PROX-10,0.067981,0.127531,0.130884,0.130884,0.130884


### Tabel Plausibility (LOF) Per Profil

In [46]:
plausibility_rows = []
for index, profile in enumerate(objective_weight_sweep_report["profiles"], start=1):
    selections = profile["selected_candidates"]
    row = {"ID": f"PLAUS-{index:02d}"}
    for config_key, column_name in objective_config_columns:
        row[column_name] = selections[config_key]["plausibility_lof"]
    plausibility_rows.append(row)

objective_plausibility_table = pd.DataFrame(plausibility_rows)
display(HTML(objective_plausibility_table.to_html(index=False)))

ID,Proximity Only,75/25,50/50,25/75,Plausibility Only
PLAUS-01,1.097106,1.097106,1.097106,1.091580,1.091580
PLAUS-02,0.973215,0.951435,0.951435,0.951435,0.951435
PLAUS-03,1.174521,1.174521,1.174521,1.174521,1.174521
PLAUS-04,1.049605,1.049605,1.049405,1.049405,1.049405
PLAUS-05,1.119080,1.119080,1.052703,1.052703,1.052703
PLAUS-06,1.462193,1.412056,1.412056,1.412056,1.412056
PLAUS-07,0.954985,0.939209,0.939209,0.939209,0.939209
PLAUS-08,1.034699,1.013437,1.013437,1.013437,1.013437
PLAUS-09,1.406638,1.406638,1.406638,1.322103,1.247344
PLAUS-10,1.000279,0.982628,0.982239,0.982239,0.982239


### Tabel Agregat

In [47]:
summary_by_objective = objective_weight_sweep_report["summary_by_objective"]
objective_summary_rows = []
for config_key, column_name in objective_config_columns:
    summary = summary_by_objective[config_key]
    objective_summary_rows.append(
        {
            "Konfigurasi": column_name,
            "Mean Proximity": summary["mean_proximity"],
            "Mean Plausibility (LOF)": summary["mean_plausibility_lof"],
        }
    )

objective_summary_table = pd.DataFrame(objective_summary_rows)
display(HTML(objective_summary_table.to_html(index=False)))

Konfigurasi,Mean Proximity,Mean Plausibility (LOF)
Proximity Only,0.280539,1.178784
75/25,0.291090,1.165053
50/50,0.298479,1.161086
25/75,0.355473,1.138299
Plausibility Only,0.448849,1.132516
